In [ ]:
#worked previously

import os
import base64
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage

# 1. Initialize the LLM
llm = ChatVertexAI(
    model="gemini-2.5-pro",
    project=os.getenv("GOOGLE_CLOUD_PROJECT"),
    location="asia-southeast1"
)

# 2. Read and Base64-Encode your local file
file_path = "../data/opium_history.txt"
with open(file_path, "rb") as f:
    encoded_content = base64.b64encode(f.read()).decode("utf-8")

# 3. Create the Multimodal Message
message = HumanMessage(
    content=[
        {"type": "text", "text": "Analyze this document and answer: What is opium?"},
        {
            "type": "media",
            "mime_type": "text/plain",
            "data": encoded_content  # Use 'data' for base64 instead of 'file_uri'
        },
    ]
)

# 4. Invoke the model
response = llm.invoke([message])
print(response.content)

/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_99449/731627960.py:9: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised NotFound: 404 Publisher Model `projects/project-3f441d35-164a-4a8d-b3a/locations/asia-southeast1/publishers/google/models/gemini-2.5-pro` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions.
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised NotFound: 404 Publisher Model `projects/project-3f441d35-164a-4a8d-b3a/locations/asia-southeast1/publishers/google/models/gemini-2.5-pro` was not found or your project does

NotFound: 404 Publisher Model `projects/project-3f441d35-164a-4a8d-b3a/locations/asia-southeast1/publishers/google/models/gemini-2.5-pro` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions

SUMMARY: collect all docs and summarise 1 docs

Limited to 10 files

In [3]:
import os
from google.cloud import storage
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage

# 1. Initialize Clients using Project ID
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
storage_client = storage.Client(project=project_id)
llm = ChatVertexAI(model="gemini-2.5-flash", project=project_id)

# 2. Find and List Text Files
all_historical_files = []  # store (bucket_name, blob_name)

buckets = list(storage_client.list_buckets())

#for testing, limit to 10 files
MAX_FILES = 10

for bucket in buckets:
    blobs = storage_client.list_blobs(bucket.name)

    for blob in blobs:
        if blob.name.endswith(".txt") and blob.size > 0:
            all_historical_files.append((bucket.name, blob.name))

            # Download and print content
            #content = blob.download_as_text()
            #uri = f"gs://{bucket.name}/{blob.name}"

            #print(f"Content of {uri}:\n")
            #print(content)
            if len(all_historical_files) >= MAX_FILES:
                break
    if len(all_historical_files) >= MAX_FILES:
        break

print(f"✅ Found {len(all_historical_files)} records in project '{project_id}'")




/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_99449/1547382498.py:9: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-flash", project=project_id)


✅ Found 10 records in project 'project-3f441d35-164a-4a8d-b3a'


In [ ]:
# 3. Summarise a specified file 
if all_historical_files:
    bucket_name, blob_name = all_historical_files[5]
    target_uri = f"gs://{bucket_name}/{blob_name}"

    print(f"🧐 Analyzing: {target_uri}")

    # Fetch the correct blob again
    blob = storage_client.bucket(bucket_name).blob(blob_name)
    content = blob.download_as_text()

    message = HumanMessage(
        content=[
            {"type": "text", "text": "Provide a historian-grade summary. What are the key justifications found?"},
            {"type": "text", "text": content}
        ]
    )

    response = llm.invoke([message])

    print("\n--- Historian Summary ---")
    print(response.content)

# ~20s



🧐 Analyzing: gs://hist-hack-bucket2/ocr_results/CO2731 - Miscellaneous_(India_Office) (1838-1858)/CO2731 - Miscellaneous_(India_Office) (1838-1858)_0006.txt

--- Historian Summary ---
This collection of reports and documents primarily by Captain/Major J. Low, dating from 1827 to 1838, offers a granular insight into the justifications employed for British colonial activities in Southeast Asia, particularly concerning the territories around Penang, Province Wellesley, and Kedah. The records reveal a consistent rationale for intervention, administration, and military action, often framed around the maintenance of order, assertion of control, and protection of British interests.

The key justifications found can be categorized as follows:

1.  **Suppression of Insurrection and Dissent:** Several entries explicitly document the quelling of "insurrection" (1827) and the "dislodg[ing]" of "insurgents and Bugis" (1838) on the Kedah frontier by military force. These reports serve to legitimize 

INTERROGATION: Retrieval based Answering

Limited to 10 files

In [ ]:
import os
from google.cloud import storage
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

storage_client = storage.Client(project=project_id)
llm = ChatVertexAI(model="gemini-2.5-flash", project=project_id)
embeddings = VertexAIEmbeddings(project=project_id, model="gemini-embedding-001")

bucket_name = "hist-hack-bucket2"


#for testing, limit to 10 files
MAX_FILES = 10

documents = []

print("📥 Loading documents from GCS...")

for blob in storage_client.list_blobs(bucket_name):
    if blob.name.endswith(".txt") and blob.size > 0:
        content = blob.download_as_text()

        documents.append(
            Document(
                page_content=content,
                metadata={"source": f"gs://{bucket_name}/{blob.name}"}
            )
        )

        if len(documents) >= MAX_FILES:
                break
    if len(documents) >= MAX_FILES:
        break

print(f"✅ Loaded {len(documents)} documents")

# 🔹 Chunk documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

chunked_docs = text_splitter.split_documents(documents)

print(f"✂️ Created {len(chunked_docs)} chunks")

# 🔹 Create FAISS vector store
vectorstore = FAISS.from_documents(chunked_docs, embeddings)

print("📦 Vector store ready.")

#~90s

/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_99449/3307174438.py:11: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-flash", project=project_id)
/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_99449/3307174438.py:12: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(project=project_id, model="gemini-embedding-001")
/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_99449/3307174438.py:12: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embe

📥 Loading documents from GCS...
✅ Loaded 10 documents
✂️ Created 12 chunks
📦 Vector store ready.


In [ ]:
# takes very long/doesnt work

query = "What justifications were used to defend British administrative control?"
print(f"🔍 Running retrieval for query: {query}")

# Test with only 1 chunk
relevant_docs = vectorstore.similarity_search(query, k=2)  # smaller k for faster testing
print(f"✅ Retrieved {len(relevant_docs)} chunks.")
for i, doc in enumerate(relevant_docs):
    print(f"- Chunk {i+1} source: {doc.metadata['source']}, length: {len(doc.page_content)}")

context = "\n\n".join([doc.page_content[:500] for doc in relevant_docs])  # truncate for test
print("📄 Sending context to LLM (truncated to 500 chars per chunk for speed)...")

message = [
    {"type": "text", "text": "Answer as a historian using ONLY the provided context."},
    {"type": "text", "text": context},
    {"type": "text", "text": f"Question: {query}"}
]

print("⏳ Invoking LLM...")
response = llm.invoke(message)
print("✅ LLM response received!")

print("\n=== HISTORIAN ANSWER ===\n")
print(response.content)

🔍 Running retrieval for query: What justifications were used to defend British administrative control?


misc

In [22]:
import os
from google.cloud import storage

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
storage_client = storage.Client(project=project_id)

bucket_name = "hist-hack-bucket2"
bucket = storage_client.bucket(bucket_name)

# List all blobs recursively
all_txt_blobs = [blob for blob in storage_client.list_blobs(bucket_name) if blob.name.endswith(".txt") and blob.size > 0]

print(f"✅ Found {len(all_txt_blobs)} text files in bucket '{bucket_name}'")

# Optional: print first few paths
for blob in all_txt_blobs[:5]:
    print(blob.name)

✅ Found 17012 text files in bucket 'hist-hack-bucket2'
ocr_results/CO2731 - Miscellaneous_(India_Office) (1838-1858)/CO2731 - Miscellaneous_(India_Office) (1838-1858)_0001.txt
ocr_results/CO2731 - Miscellaneous_(India_Office) (1838-1858)/CO2731 - Miscellaneous_(India_Office) (1838-1858)_0002.txt
ocr_results/CO2731 - Miscellaneous_(India_Office) (1838-1858)/CO2731 - Miscellaneous_(India_Office) (1838-1858)_0003.txt
ocr_results/CO2731 - Miscellaneous_(India_Office) (1838-1858)/CO2731 - Miscellaneous_(India_Office) (1838-1858)_0004.txt
ocr_results/CO2731 - Miscellaneous_(India_Office) (1838-1858)/CO2731 - Miscellaneous_(India_Office) (1838-1858)_0005.txt


In [23]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# 1. Point this to the folder containing your high-fidelity OCR output
loader = DirectoryLoader(
    "../data", 
    glob="**/*.txt", 
    loader_cls=TextLoader
)

# 2. This defines 'docs' as a list of Document objects
docs = loader.load()

In [1]:
from langchain_community.document_loaders import TextLoader
# -----------------------------
# 1. Load your corpus
# -----------------------------
docs = []
loader = TextLoader("../data/opium_history.txt")  # or iterate over multiple files
docs = loader.load()


/Users/janelle/Downloads/projects/hist-hack/.venv12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/janelle/Downloads/projects/hist-hack/.venv12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 1. Setup the Brain (Model & Embeddings)
# Use the latest 2026 stable models
llm = ChatVertexAI(model="gemini-2.5-pro", project=os.getenv("GOOGLE_CLOUD_PROJECT"),  max_thought_tokens=1024,                   # Standard 2026 parameter for budget
    location="asia-southeast1", include_thought=False) # Disables the thinking process for speed)
# Add this to your ChatVertexAI initialization
embeddings = VertexAIEmbeddings(model="text-embedding-004")


# 2. Build the Vector Store (The Corpus)
# 'docs' should be a list of LangChain Document objects from your OCR
# For now, we'll use a local FAISS index (perfect for historical research)

print(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))


# Point to your file explicitly
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Initialize embeddings (it will now 'see' the environment variable)
embeddings = VertexAIEmbeddings(model="text-embedding-004")

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})



/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_37197/785640032.py:14: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-pro", project=os.getenv("GOOGLE_CLOUD_PROJECT"),  max_thought_tokens=1024,                   # Standard 2026 parameter for budget
/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_37197/785640032.py:14: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(model="gemini-2.5-pro", project=os.getenv("GOOGLE_CLOUD_PROJECT"),  max_thought_tokens=1024,                   # Standard 2026 parameter for budget
Unexpected argument 'max_thought_to

../application_default_credentials.json


In [5]:
# 3. Define the "Historian" Prompt
template = """
You are a senior colonial historian. Use the following archival fragments 
to answer the question. If the answer isn't in the text, say you don't know.

Context: {context}

Question: {question}

Answer with archival citations:
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Create the RAG Chain (The LCEL way)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)




In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [4]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = model.invoke(messages)
ai_msg

AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-pro', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ca8f1-f1aa-7ea2-aaf7-e165295d56fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 4288, 'total_tokens': 4309, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 4281}})

In [11]:
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import os
from dotenv import load_dotenv

load_dotenv()

# 1. Setup your VLM
llm = ChatVertexAI(model="gemini-2.5-pro")

# 2. Define your "Historian-Grade" Prompt
prompt = ChatPromptTemplate.from_template("""
Write a historian-grade summary of the following colonial documents. 
Focus on key entities, dates, and the underlying thematic justifications.
Context: {context}
""")

# 3. Create the modern chain
summarize_chain = create_stuff_documents_chain(llm, prompt)

# 4. Run it
# result = summarize_chain.invoke({"context": docs})

/var/folders/r1/6yg09d_104x7y93zb29f0m_w0000gn/T/ipykernel_2677/3297534655.py:11: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(model="gemini-2.5-pro")
Failed to convert project number to project ID.
Traceback (most recent call last):
  File "/Users/janelle/Downloads/projects/hist-hack/.venv12/lib/python3.12/site-packages/google/api_core/grpc_helpers.py", line 55, in error_remapped_callable
    return callable_(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/janelle/Downloads/projects/hist-hack/.venv12/lib/python3.12/site-packages/grpc/_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ^^^^^^^^^^^^^^^^
  File "/Users/janelle/Downloads/projects/hist-hack/.venv12/lib/python3.12/site-packages/grpc/_interceptor.py", line 331, in _with_call
    return call.result(), call
           ^^^^^^^^^^^^^
  File "/Users/janelle/Downloa

In [ ]:
from langchain_google_vertexai import ChatVertexAI
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document

# Initialize Gemini 1.5 Pro
llm = ChatVertexAI(model_name="gemini-1.5-pro", temperature=0.2)

# Custom prompt for historical accuracy
historical_prompt = """
You are a senior archivist. Summarize the following colonial despatch. 
Identify the primary sender, recipient, and the specific colonial concern (e.g., revenue, unrest).
Highlight any 'marginalia' or clerk notes mentioned in the text.
Text: {text}
"""

# Load the "refine" chain (best for complex, multi-page despatches)
chain = load_summarize_chain(llm, chain_type="refine")
# result = chain.run(docs)